# Deep Learning Assignment: One-Shot $S \rightarrow \Delta M$ Mapping

**Release: Student Version**  
**Due date: See the course platform**

This assignment concerns an anonymous dual-band two-port system identified only as **CM19**. You are given its simulator, legal parameter bounds, and compliance requirements. You are **not** given a training dataset or a prescribed solution method.

Your task is to design and train a model that maps the current complex frequency response $S$ directly to a 41-dimensional correction $\Delta m$, representing the tunable entries of $\Delta M$:

$$f_\theta: S \longmapsto \Delta m, \qquad M_{\mathrm{corrected}}=M_{\mathrm{current}}+\Delta M.$$

The correction is applied once. The corrected response should satisfy every published requirement.

## 1. What Is Provided

You receive only:

1. a deterministic CM19 simulator;
2. a reproducible detuned-state generator;
3. lower and upper bounds for the 41 tunable entries;
4. the seven compliance requirements below; and
5. an evaluation and export interface.

You must design everything else, including how to explore the simulator, obtain learning targets or another training signal, choose a model, define an objective, and validate the final mapping.

## 2. Compliance Requirements

A corrected state is compliant only when **all seven** conditions hold. Values are magnitude in dB. Every condition is evaluated over all simulator frequency samples inside the stated closed interval.

| Trace | Frequency range (MHz) | Requirement |
|---|---:|---:|
| $S_{11}$ | 1804–1881 | $\max S_{11}\leq-20.00$ dB |
| $S_{11}$ | 2109–2171 | $\max S_{11}\leq-20.00$ dB |
| $S_{21}$ | 1805–1807.5 | $\min S_{21}\geq-1.53$ dB |
| $S_{21}$ | 1807.5–1877.5 | $\min S_{21}\geq-1.28$ dB |
| $S_{21}$ | 1877.5–1880 | $\min S_{21}\geq-1.20$ dB |
| $S_{21}$ | 2110–2167 | $\min S_{21}\geq-1.25$ dB |
| $S_{21}$ | 2167.5–2170 | $\min S_{21}\geq-1.28$ dB |

In [ ]:
%matplotlib inline

from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

try:
    from assignments.cm19_simulator import CM19Simulator
except ImportError:
    from cm19_simulator import CM19Simulator

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("submission_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

simulator = CM19Simulator()
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Response shape:", simulator.response_shape)
print("Correction dimension:", simulator.correction_dim)
print("Frequency points:", len(simulator.frequency_hz))
simulator.print_requirements()

## 3. Public Simulator Interface

The simulator uses the following contract:

- `sample_state(seed) -> (m_current, response)` returns a reproducible detuned state.
- `simulate(m_vector) -> response` evaluates any legal 41-vector.
- `check_response(response) -> dict` evaluates all seven requirements.
- `apply_correction(m_current, delta_m) -> CorrectionResult` clips the corrected vector to legal bounds, simulates it, and reports compliance.
- `lower_bounds` and `upper_bounds` define the legal range of every tunable entry.

The response has shape `(4, 2001)` with channel order:

```text
[Re(S11), Im(S11), Re(S21), Im(S21)]
```

The final submitted model must accept a batch with shape `(B, 4, 2001)` and return `delta_m` with shape `(B, 41)`.

In [ ]:
m0, s0 = simulator.sample_state(seed=0)
initial_info = simulator.check_response(s0)

print("m_current shape:", m0.shape)
print("response shape:", s0.shape)
print("within bounds:", np.all(m0 >= simulator.lower_bounds) and np.all(m0 <= simulator.upper_bounds))
print("all specs pass:", initial_info["all_specs_pass"])
print("maximum specification miss (dB):", initial_info["max_spec_miss_db"])

In [ ]:
def plot_response(response, title="CM19 response"):
    s11_db, s21_db = simulator.response_db(response)
    frequency_mhz = simulator.frequency_hz / 1e6
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.plot(frequency_mhz, s11_db, label="S11", linewidth=1.2)
    ax.plot(frequency_mhz, s21_db, label="S21", linewidth=1.2)
    for req in simulator.requirements:
        color = "tab:blue" if req.trace == "S11" else "tab:orange"
        ax.hlines(req.threshold_db, req.start_mhz, req.stop_mhz, color=color,
                  linewidth=3, alpha=0.35)
    ax.set(xlabel="Frequency (MHz)", ylabel="Magnitude (dB)", title=title)
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()

plot_response(s0, title="Example detuned state (seed 0)")

## 4. Your Solution Starts Here

There is deliberately no reference search procedure, target-generation code, loss function, or network architecture in this notebook. Your report must explain and justify your complete solution pipeline.

The simulator may be called as often as your compute budget permits. You may formulate the problem using supervised learning, self-supervision, simulator-in-the-loop learning, two-stage methods, or another defensible deep-learning approach. Regardless of the training method, the final deliverable must be a single one-shot $S\rightarrow\Delta m$ model.

### 4.1 Simulator Exploration and Problem Formulation (Required)

Before training a model:

1. evaluate at least 20 detuned seeds;
2. summarize the initial compliance rate and per-requirement misses;
3. examine the parameter bounds and response variability;
4. obtain at least one compliant parameter vector using a method you design; and
5. explain how your method will provide a useful learning signal for $S\rightarrow\Delta m$.

Document the number of simulator calls and wall-clock time used.

In [ ]:
# TODO 1: Explore the simulator, quantify the starting difficulty, and find at least
# one compliant vector. Add cells, tables, and plots as needed.
raise NotImplementedError("Complete TODO 1: simulator exploration and formulation")

### 4.2 Construct a Learning Signal (Required)

Design the procedure that supplies supervision or another training signal. Address:

- how current states are selected;
- how candidate corrections are obtained;
- what makes a correction a valid target when multiple compliant corrections may exist;
- how simulator calls are budgeted;
- how train/validation states are separated by seed; and
- how you prevent information from validation or public evaluation seeds entering training.

In [ ]:
# TODO 2: Implement your learning-signal or target-construction procedure.
# Do not use the public evaluation seeds defined later in this notebook.
raise NotImplementedError("Complete TODO 2: construct a learning signal")

## 5. Implement the One-Shot Mapping

Your final model must satisfy all of the following:

- raw input shape `(B, 4, 2001)`;
- output shape `(B, 41)` in the original matrix-parameter scale;
- no simulator calls inside `forward`;
- no seed, current matrix vector, or handcrafted state identifier as input;
- no more than **20 million** trainable parameters; and
- finite output for every finite simulator response.

Any normalization required by the mapping must be stored inside the exported model.

In [ ]:
class SToDeltaM(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 3: Define your model, including any fixed input/output normalization.
        self.network = None

    def forward(self, response):
        # response: (B, 4, 2001); return delta_m: (B, 41)
        if self.network is None:
            raise NotImplementedError("Complete TODO 3: model architecture")
        return self.network(response)

model = SToDeltaM().to(DEVICE)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")
assert trainable_params <= 20_000_000, "The model exceeds the 20M-parameter limit"

### 5.1 Objective and Training (Required)

Define and justify the learning objective. If you use multiple loss terms, explain the units, scaling, and weights. Record the random seed, optimizer, learning-rate schedule, batch size, stopping rule, simulator-call count, and training time. Save the model selected on validation states rather than the final epoch automatically.

In [ ]:
# TODO 4: Implement the objective, training loop, validation, checkpoint selection,
# and learning curves.
raise NotImplementedError("Complete TODO 4: objective and training")

## 6. Standard One-Shot Evaluation

The primary metric is the fraction of detuned states that satisfy **all seven requirements after one predicted correction**. The secondary metric is the mean and median maximum specification miss in dB.

Seeds `10000` through `10049` are public evaluation seeds. Do not use them for training, target construction, normalization, model selection, or hyperparameter tuning. The final grade uses different hidden seeds.

In [ ]:
@torch.no_grad()
def evaluate_one_shot(model, seeds):
    model.eval()
    rows = []
    for seed in seeds:
        current_m, response = simulator.sample_state(int(seed))
        x = torch.from_numpy(response[None]).to(DEVICE)
        delta_m = model(x).detach().cpu().numpy()[0]
        if delta_m.shape != (41,) or not np.isfinite(delta_m).all():
            rows.append({"seed": int(seed), "passed": False, "max_miss_db": float("inf")})
            continue
        result = simulator.apply_correction(current_m, delta_m)
        rows.append({
            "seed": int(seed),
            "passed": result.all_specs_pass,
            "max_miss_db": result.max_spec_miss_db,
            "correction_l2": float(np.linalg.norm(delta_m)),
        })

    finite_misses = np.asarray([row["max_miss_db"] for row in rows], dtype=np.float64)
    summary = {
        "num_states": len(rows),
        "pass_rate": float(np.mean([row["passed"] for row in rows])),
        "mean_max_miss_db": float(np.mean(finite_misses)),
        "median_max_miss_db": float(np.median(finite_misses)),
    }
    return summary, rows

PUBLIC_EVALUATION_SEEDS = np.arange(10000, 10050)
# public_summary, public_rows = evaluate_one_shot(model, PUBLIC_EVALUATION_SEEDS)
# print(public_summary)

### 6.1 Required Analysis

Report:

1. the public one-shot pass rate;
2. mean, median, and worst maximum specification miss;
3. pass/fail counts for each individual requirement;
4. at least three corrected-response plots, including a failure case if one exists (otherwise explicitly state that all evaluated states passed);
5. correction magnitude and bound-clipping statistics;
6. one meaningful ablation; and
7. a discussion of failure modes and the ambiguity of mapping one response to a correction.

In [ ]:
# TODO 5: Run the standard evaluation and complete the required analysis.
raise NotImplementedError("Complete TODO 5: evaluation and analysis")

## 7. Export the Final Mapping

The exported artifact must be a standalone TorchScript model. It must map a raw simulator response directly to a correction without external normalization files or simulator calls.

The helper below uses `torch.jit.trace`. If your model has input-dependent branches or loops, use an appropriate TorchScript export such as `torch.jit.script` and check agreement with the original model on several distinct responses and batch sizes.

In [ ]:
def export_model(model, output_dir=OUTPUT_DIR):
    model = model.to("cpu").eval()
    _, example_response = simulator.sample_state(seed=99999)
    example = torch.from_numpy(example_response[None])
    with torch.no_grad():
        output = model(example)
    if output.shape != (1, 41) or output.dtype != torch.float32 or not torch.isfinite(output).all():
        raise ValueError("Export check failed: expected finite float32 output with shape (1,41)")

    traced = torch.jit.trace(model, example)
    model_path = output_dir / "s_to_delta_m.pt"
    traced.save(str(model_path))
    metadata = {
        "input_shape": [None, 4, 2001],
        "output_shape": [None, 41],
        "channel_order": list(simulator.response_channels),
        "seed": SEED,
        "parameter_count": sum(p.numel() for p in model.parameters()),
    }
    metadata_path = output_dir / "metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return model_path, metadata_path

# Load your selected checkpoint before uncommenting the following line.
# model_path, metadata_path = export_model(model)

In [ ]:
def check_export(model_path):
    loaded = torch.jit.load(str(model_path), map_location="cpu").eval()
    current_m, response = simulator.sample_state(seed=123456)
    with torch.no_grad():
        delta_m = loaded(torch.from_numpy(response[None]))
    assert delta_m.shape == (1, 41)
    assert delta_m.dtype == torch.float32
    assert torch.isfinite(delta_m).all()
    result = simulator.apply_correction(current_m, delta_m.numpy()[0])
    print("Export check passed")
    print("Example corrected pass:", result.all_specs_pass)
    print("Example maximum miss (dB):", result.max_spec_miss_db)

# check_export(OUTPUT_DIR / "s_to_delta_m.pt")

## 8. Submission

Submit `studentID_name_S2DM.zip` containing only:

1. a fully executed notebook named `studentID_name.ipynb`;
2. `submission_artifacts/s_to_delta_m.pt`;
3. `submission_artifacts/metadata.json`; and
4. a report of no more than four pages describing your formulation, simulator budget, model, experiments, ablation, and failure analysis.

Do not submit generated intermediate samples, caches, all checkpoints, or external source code.

## 9. Grading Rubric (100 points)

| Category | Points | Criteria |
|---|---:|---|
| Problem formulation and simulator exploration | 15 | Correct understanding, quantitative exploration, compliant example, and simulator-budget accounting |
| Learning-signal design | 15 | Defensible construction, no seed leakage, ambiguity addressed, and validation protocol justified |
| Model and training | 20 | Correct interface, sound deep-learning design, reproducible training, and meaningful ablation |
| Public evaluation and analysis | 10 | All required metrics, plots, clipping statistics, and failure analysis |
| Hidden one-shot compliance rate | 30 | Fraction of hidden states satisfying all seven requirements after one correction |
| Hidden maximum-miss performance | 5 | Mean and median maximum specification miss on hidden states |
| Export and report quality | 5 | Standalone artifact loads on CPU; submission is complete and clearly documented |

A model that accesses the simulator in `forward`, accepts the current matrix or seed as input, uses public evaluation seeds during development, cannot be loaded, or produces non-finite outputs receives no credit for the affected sections.

## 10. Academic Integrity

You may consult public textbooks, papers, and framework documentation. Your report must disclose external sources and generative tools. You may discuss general ideas, but you may not exchange generated targets, compliant vectors, model weights, or core implementation code. Do not modify the simulator, rely on undocumented internals, recover removed engineering provenance, or attempt to obtain hidden evaluation seeds.